In [ ]:


"""
================================================================================
LAB: Student Record Management System
File: Day1_Lab_RecordManager_starter.py
--------------------------------------------------------------------------------
Trainer Section (First 30%):
  - In-memory database initialization
  - File loading logic with JSON exception handling
  - View all records formatted output
  - Interactive CLI loop scaffold

Student Completion Tasks (Remaining 70%):
  1. Complete add_student_record() with input validation
  2. Implement search_student_record() by ID or Name substring
  3. Implement delete_student_record() with confirmation
  4. Implement update_student_record()
  5. Implement save_records_to_json() with safe file flushing
  6. STRETCH GOAL: Implement export_to_csv()
================================================================================
"""
import csv
import json
import os
import sys
from typing import Dict, Any

# Target data file
DATABASE_FILE = "sample_records.json"

# In-memory storage: Key = Student ID, Value = Dict of Student attributes
STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Loads student records safely from a JSON file.

    Handles FileNotFoundError and corrupted JSON formatting gracefully.
    """
    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data
    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}
    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""
    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75
    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)
    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)
        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")
    print(separator + "\n")


# ==============================================================================
# ✍️ STUDENT TASKS TO IMPLEMENT BELOW (TODO SECTION)
# ==============================================================================

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:

    print("\n--- Add New Student ---")

    student_id = input("Enter Student ID: ").strip()

    if not student_id:
        print("[ERROR] Student ID cannot be empty.")
        return

    if student_id in registry:
        print("[ERROR] Student ID already exists.")
        return

    # Name
    name = input("Enter Student Name: ").strip()

    if not name:
        print("[ERROR] Name cannot be empty.")
        return

    # Branch
    branch = input("Enter Branch: ").strip()

    if not branch:
        print("[ERROR] Branch cannot be empty.")
        return

    # CGPA
    cgpa_input = input("Enter CGPA (0.0 - 10.0): ").strip()

    try:
        cgpa = float(cgpa_input)

        if cgpa < 0.0 or cgpa > 10.0:
            print("[ERROR] CGPA must be between 0.0 and 10.0.")
            return

    except ValueError:
        print("[ERROR] CGPA must be a valid number.")
        return

    # Generate email
    first_name = name.split()[0].lower()
    email = f"{first_name}.{student_id.lower()}@university.edu"

    # Add record
    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email
    }

    print(f"[SUCCESS] Student '{name}' added successfully.")
    print(f"[INFO] Generated email: {email}")

    pass


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 2: Search by student ID (exact) or student Name (case-insensitive substring)."""
    print("\n--- Search Student Records ---")
    if not registry:
        print("[INFO] No records available to search.")
        return

    search_term = input("Enter Student ID or Name: ").strip()

    if not search_term:
        print("[ERROR] Search term cannot be empty.")
        return

    # Exact Student ID search
    if search_term in registry:
        details = registry[search_term]

        print("\n[SUCCESS] Student Found")
        print("-" * 40)
        print(f"Student ID : {search_term}")
        print(f"Name       : {details.get('name', 'N/A')}")
        print(f"Branch     : {details.get('branch', 'N/A')}")
        print(f"CGPA       : {details.get('cgpa', 0.0):.2f}")
        print(f"Email      : {details.get('email', 'N/A')}")
        print("-" * 40)
        return

    # Case-insensitive substring name search
    search_lower = search_term.lower()
    found = False

    for student_id, details in registry.items():
        name = details.get("name", "")

        if search_lower in name.lower():
            if not found:
                print("\nMatching Records:")
                print("-" * 75)
                print(
                    f"{'Student ID':<12} | "
                    f"{'Name':<22} | "
                    f"{'Branch':<22} | "
                    f"{'CGPA':<5}"
                )
                print("-" * 75)

            print(
                f"{student_id:<12} | "
                f"{name:<22} | "
                f"{details.get('branch', 'N/A'):<22} | "
                f"{details.get('cgpa', 0.0):<5.2f}"
            )

            found = True

    if found:
        print("-" * 75)
    else:
        print("[INFO] No matching student records found.")

    pass


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 3: Prompt for Student ID and delete record with confirmation."""
    print("\n--- Delete Student Record ---")
    if not registry:
        print("[INFO] No records available to delete.")
        return

    student_id = input("Enter Student ID to delete: ").strip()

    if student_id not in registry:
        print("[ERROR] Student ID not found.")
        return

    student = registry[student_id]

    print("\nStudent record:")
    print(f"Name   : {student.get('name', 'N/A')}")
    print(f"Branch : {student.get('branch', 'N/A')}")
    print(f"CGPA   : {student.get('cgpa', 0.0):.2f}")

    confirmation = input("Are you sure you want to delete this record? (y/n): ").strip().lower()

    if confirmation in ("y", "yes"):
        del registry[student_id]
        print("[SUCCESS] Student record deleted successfully.")
    else:
        print("[INFO] Delete operation cancelled.")


def save_records_to_json(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    pass


def save_records_to_json(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 4: Serialize the in-memory registry dictionary to the JSON file safely."""
    try:
        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(registry, file, indent=2)

        print(f"[SUCCESS] Saved {len(registry)} record(s) to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to save records: {err}")


def export_to_csv(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    pass


def export_to_csv(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 5 (Bonus): Export all student records to a CSV file."""
    if not registry:
        print("[INFO] No records available to export.")
        return

    try:
        fieldnames = ["student_id", "name", "branch", "cgpa", "email"]

        with open(file_path, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)

            writer.writeheader()

            for student_id, details in registry.items():
                writer.writerow({
                    "student_id": student_id,
                    "name": details.get("name", ""),
                    "branch": details.get("branch", ""),
                    "cgpa": details.get("cgpa", 0.0),
                    "email": details.get("email", "")
                })

        print(f"[SUCCESS] Exported {len(registry)} record(s) to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to export records: {err}")
    pass


def main_menu() -> None:
    """Main CLI control loop."""
    global STUDENT_REGISTRY
    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit
========================================
"""
    while True:
        print(menu_banner)
        choice = input("Enter choice [0-6]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)
        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)
        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)
        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)
        elif choice == "5":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
        elif choice == "6":
            export_to_csv("students_export.csv", STUDENT_REGISTRY)
        elif choice == "0":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
            print("[INFO] Application closed successfully. Good bye!")
            sys.exit(0)
        else:
            print("[WARN] Invalid option selected. Please enter a number between 0 and 6.")


if __name__ == "__main__":
    main_menu()


[WARN] Database file 'sample_records.json' not found. Starting with empty registry.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 2023250701
[WARN] Invalid option selected. Please enter a number between 0 and 6.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 1

[INFO] No records found in the registry.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 2

--- Add New Student ---
Enter Student ID: 2023250701
Enter Student Name: Vaishnavi
Enter Branch: cse
Enter CGPA (0.0 - 10.0): 9
[SUCCESS] Student 'Vaishnavi' added succe